# Bronze Layer Ingestion

This notebook implements the **Bronze layer** of the Medallion Architecture for the AgentOps Control Tower.

It reads the seven OneLake shortcuts created in Challenge 3 and preserves each physical source in its own Delta table:
- FOCUS cost and Azure Resource Graph metadata from Parquet.
- Application requests, dependencies, and metrics from newline-delimited JSON.
- Azure Monitor audit logs and platform metrics from JSON.

Bronze ingestion adds source-file and ingestion-time lineage columns. Source column names are made Delta-safe by replacing unsupported punctuation with underscores; source values are not transformed. Cleaning, normalization, and cross-source joins belong in Silver.


In [ ]:
import re

from pyspark.sql.functions import col, count, current_timestamp, input_file_name, isnull, when

spark.conf.set("spark.sql.parquet.mergeSchema", "true")


def sanitize_delta_columns(df):
    safe_columns = []
    used_columns = set()
    renamed_columns = []

    for original_name in df.columns:
        base_name = re.sub(r"[^A-Za-z0-9_]", "_", original_name)
        base_name = re.sub(r"_+", "_", base_name).strip("_") or "column"
        if base_name[0].isdigit():
            base_name = f"_{base_name}"

        safe_name = base_name
        suffix = 2
        while safe_name.lower() in used_columns:
            safe_name = f"{base_name}_{suffix}"
            suffix += 1

        used_columns.add(safe_name.lower())
        safe_columns.append(safe_name)
        if safe_name != original_name:
            renamed_columns.append((original_name, safe_name))

    if renamed_columns:
        print(f"Sanitized {len(renamed_columns)} Delta column name(s):")
        for original_name, safe_name in renamed_columns:
            print(f"  {original_name} -> {safe_name}")

    return df.toDF(*safe_columns)


def read_bronze_source(path, file_format, path_glob_filter=None):
    reader = spark.read.format(file_format).option("recursiveFileLookup", "true")

    if file_format == "parquet":
        reader = reader.option("mergeSchema", "true")
    elif file_format == "json":
        reader = reader.option("multiLine", "false").option("mode", "PERMISSIVE")

    if path_glob_filter:
        reader = reader.option("pathGlobFilter", path_glob_filter)

    source_df = sanitize_delta_columns(reader.load(path))
    return (
        source_df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )


## 1. Ingest FOCUS Cost Data

Read FOCUS-compliant Parquet exports from `Files/costs/`. Schema merging accommodates additive changes across billing periods.


In [ ]:
df_costs_raw = read_bronze_source(
    "Files/costs/",
    "parquet",
    "*.parquet",
)

print(f"Cost records loaded: {df_costs_raw.count()}")
df_costs_raw.printSchema()


## 2. Ingest Resource Graph Metadata

Read only the full Resource Graph snapshot from `Files/metadata/`. The shortcut also contains aggregate summary Parquet files with different schemas, so `pathGlobFilter` prevents them from being merged into the resource table.


In [ ]:
df_resource_metadata_raw = read_bronze_source(
    "Files/metadata/",
    "parquet",
    "all_resources_with_tags_*.parquet",
)

print(f"Resource metadata records loaded: {df_resource_metadata_raw.count()}")
df_resource_metadata_raw.printSchema()


## 3. Ingest Application Telemetry

Read the three Log Analytics data-export feeds as newline-delimited JSON. Keeping requests, dependencies, and custom metrics separate preserves their source schemas for Silver processing.


In [ ]:
df_apprequests_raw = read_bronze_source("Files/telemetry/apprequests/", "json")
df_appdependencies_raw = read_bronze_source("Files/telemetry/appdependencies/", "json")
df_appmetrics_raw = read_bronze_source("Files/telemetry/appmetrics/", "json")

print(f"Application request records loaded: {df_apprequests_raw.count()}")
print(f"Application dependency records loaded: {df_appdependencies_raw.count()}")
print(f"Application metric records loaded: {df_appmetrics_raw.count()}")


## 4. Ingest Platform Diagnostics

Read Azure Monitor diagnostic-setting output from the audit-log and platform-metric shortcuts. Azure writes these hourly blobs as JSON event records.


In [ ]:
df_diagnostic_audit_raw = read_bronze_source("Files/diagnostics/audit/", "json")
df_platform_metrics_raw = read_bronze_source("Files/diagnostics/platformmetrics/", "json")

print(f"Diagnostic audit records loaded: {df_diagnostic_audit_raw.count()}")
print(f"Platform metric records loaded: {df_platform_metrics_raw.count()}")


## 5. Data Quality Checks

Before writing Delta, report row counts, expected columns, and malformed JSON records. Missing columns remain warnings because Bronze preserves source data without applying Silver transformations.


In [ ]:
def run_quality_checks(df, table_name, expected_columns):
    row_count = df.count()
    print(f"\n--- Quality Report: {table_name} ---")
    print(f"Total rows: {row_count}")
    print(f"Total columns: {len(df.columns)}")

    missing_columns = [column for column in expected_columns if column not in df.columns]
    if missing_columns:
        print(f"WARNING: Missing expected columns: {missing_columns}")

    present_columns = [column for column in expected_columns if column in df.columns]
    if row_count > 0 and present_columns:
        null_expressions = [
            count(when(isnull(col(column)), column)).alias(f"{column}_nulls")
            for column in present_columns
        ]
        null_counts = df.select(null_expressions).first().asDict()
        for column in present_columns:
            null_count = null_counts[f"{column}_nulls"]
            print(f"  {column}: {null_count} nulls")

    if "_corrupt_record" in df.columns:
        corrupt_count = df.filter(col("_corrupt_record").isNotNull()).count()
        print(f"  Malformed JSON records: {corrupt_count}")

    return row_count


bronze_tables = {
    "bronze_costs": df_costs_raw,
    "bronze_resource_metadata": df_resource_metadata_raw,
    "bronze_apprequests": df_apprequests_raw,
    "bronze_appdependencies": df_appdependencies_raw,
    "bronze_appmetrics": df_appmetrics_raw,
    "bronze_diagnostic_audit": df_diagnostic_audit_raw,
    "bronze_platform_metrics": df_platform_metrics_raw,
}

expected_columns = {
    "bronze_costs": ["BilledCost", "EffectiveCost", "ServiceName", "ChargePeriodStart"],
    "bronze_resource_metadata": ["id", "name", "type", "resourceGroup", "subscriptionId"],
    "bronze_apprequests": ["TimeGenerated", "Name"],
    "bronze_appdependencies": ["TimeGenerated", "Name"],
    "bronze_appmetrics": ["TimeGenerated", "Name"],
    "bronze_diagnostic_audit": ["time", "resourceId", "category", "operationName"],
    "bronze_platform_metrics": ["time", "resourceId", "metricName"],
}

quality_results = {
    table_name: run_quality_checks(df, table_name, expected_columns[table_name])
    for table_name, df in bronze_tables.items()
}


## 6. Write Bronze Delta Tables

Persist the seven raw datasets as Delta tables under the Lakehouse `Tables/dbo/` area. This workshop notebook uses overwrite mode for repeatable full-refresh runs.


In [ ]:
for table_name, df in bronze_tables.items():
    table_path = f"Tables/dbo/{table_name}"
    print(f"Writing {table_name} to {table_path}...")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )
    written_count = spark.read.format("delta").load(table_path).count()
    print(f"  {table_name}: {written_count} rows written successfully.")

print("\nBronze layer ingestion complete.")
